# Lab: Build a TOR-Like Client

In this lab, you will build a client that mimics the functionality of the TOR network. Your objective is to create a client that forms a circuit between different nodes, encrypts information in multiple layers, and ensures secure and anonymous communication.

### Objectives:

1. **Forming the Circuit**: The client needs to form the circuit between different nodes. This involves discovering the nodes on the network and obtaining their public keys.
   
2. **Encrypting Information**: The client will need to encrypt the information multiple times (minimum of 2), with each layer encrypted using the public key of the corresponding node in the circuit.
   
3. **Sending Data to the First Node**: The client will send the encrypted data to the first node in the circuit.

### Be aware of this while building the client. This is how the server should work. (Note: The server is part 3)

4. **Node Decryption and Forwarding**: When a node decrypts its layer of encryption, it will reveal:
   - The remaining encrypted information, which the node will pass to the next node in the circuit.
   - The IP address and port of the next node in the circuit.

5. **Handling the Final Node**: If the node is the last in the circuit, upon decryption, the packet will contain the HTTP request to be sent to the target server. The response from the target server should be sent back to the client using the same circuit.

In [ ]:
import socket
import threading
import os
import ssl
import base64
import json
import struct
from Crypto.PublicKey import RSA
from time import sleep
from Crypto.Cipher import PKCS1_OAEP, AES
from Crypto.Random import get_random_bytes
from Crypto.Hash import SHA256
# By default PKCS1_OAEP uses SHA1, so I am manually setting it to SHA256

def AddNodetoJson(node_name, public_key, port):
    with open("servers.json", "r") as file:
        data = json.load(file)

    data["servers"][node_name] = {"public_key": public_key, "port": port}

    with open("directory.json", "w") as file:
        json.dump(data, file, indent=4)

def AddClienttoJson(client_name, public_key, port):
    with open("servers.json", "r") as file:
        data = json.load(file)

    data["users"][client_name] = {"public_key": public_key, "port": port}

    with open("directory.json", "w") as file:
        json.dump(data, file, indent=4)

#This can be a mockup and can hardcode this
def GetNodeInfo(server_name):
    with open("servers.json", "r") as file:
        data = json.load(file)
    
    server = data["servers"][server_name]

    server_address = server["server_address"]
    port = int(server["port"])
    server_public_key = RSA.import_key(server["public_key"])

    return server_address, port, server_public_key

def CreateCircuit():
    with open("servers.json", "r") as file:
        data = json.load(file)

    circuit = []

    # circuit should look like this [server0, server1, server2]
    for server_name in data["servers"].keys():
        circuit.append(server_name)
    
    return circuit

# Generate RSA key pair. Maybe you'll need this
def generate_rsa_key_pair():
    # Generate RSA key pair
    key = RSA.generate(2048)
    private_key = key.export_key()
    public_key = key.publickey().export_key()

    # Store keys in variables
    private_key_var = RSA.import_key(private_key)
    public_key_var = RSA.import_key(public_key)
    return private_key_var, public_key_var

# Encrypt the message with one layer of RSA encryption using the server's public key
def encrypt_message(public_key, message):
    cipher = PKCS1_OAEP.new(public_key, hashAlgorithm=SHA256)
    ciphertext = cipher.encrypt(message)
    return ciphertext
    
    
#onion_encrypts the plaintext
def onion_encrypt(message, circuit):
    # iterate through a list of public keys that should be added in order of the node being traversed
    for node in reversed(circuit):
        #use "|" as the delimiter
        #the format that is encrypted is server_ip|port|message
        server_address, port, server_public_key = GetNodeInfo(node)
        server_info = f"{server_address}|{port}|".encode()
        if node == circuit[-1]:
            message = encrypt_message(server_public_key, server_info + message)
        else:
            message = encrypt_message(server_public_key, server_info) + message
    return message           

# Decrypt the RSA encrypted message using the client's private key
def decrypt(cl_private_key, ciphertext):
    decrypt_cipher = PKCS1_OAEP.new(cl_private_key, hashAlgorithm=SHA256)
    plaintext = decrypt_cipher.decrypt(ciphertext)
    return plaintext

def recv_all(sock, response_len):
    response = b""
    while len(response) < response_len:
        chunk = sock.recv(response_len - len(response))
        if not chunk:
            raise ConnectionError("WARNING: Socket closed unexpectedly.")
        response += chunk
    return response

def recv_message(sock):
    # We are using length headers so the first four bytes will always be the length
    r_len = recv_all(sock, 4)
    message_len = struct.unpack("!I", r_len)[0] #stuct unpack returns a tuple so only need first value in tuple
    return recv_all(sock, message_len)

DESTINATION_HOST = "www.google.com"

def main():
    request = f"GET / HTTP/1.1\r\nHost: {DESTINATION_HOST}\r\n\r\n".encode()
    
    # Generate my own RSA key pair
    cl_private_key, cl_public_key = generate_rsa_key_pair()

    # Add client to a user directory so server knows the correct public key
    serial_pub = cl_public_key.export_key().decode()
    AddClienttoJson("client0", serial_pub, 9000)

    # Create the circuit
    circuit = CreateCircuit()

    # Encrypt in multi-layer encryption the http request
    encrypted_message = onion_encrypt(request, circuit)


    # Send the request to the node using sockets
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_addr, server_port, server_pubK = GetNodeInfo(circuit[0])

    sock.connect((server_addr, server_port))
    # Adding a message length header
    sock.sendall(struct.pack("!I", len(encrypted_message)) + encrypted_message)

    # Receiving the response from the node I sent it to 
    response = recv_message(sock)

    # Decrypt the response if you need to (you probably wont, but just in case you use it)
    #response = decrypt(cl_private_key, response)

    # Print the response
    print("Reponse: ", response)

    # Closing the socket to end the connection
    sock.close()
